## Create Final Table
The cleaned data mixes course information and enrollment details. This creates not only redundancy but also several issues. For example:

In the future, we want to implement a feature to index courses by professor name. However, some courses are taught by multiple professors, which makes it difficult to differentiate separate course offerings.

We might also add course rating data for different quarters in the future. Integrating this into the current table would make it very cumbersome.

Therefore, we've decided to split the current cleaned table into four separate tables:

### Table Details

#### `professors`

- **Description**: Stores information about professors.
- **Fields**:
    - **`prof_id`** (`varchar(255)`, **Primary Key**, `NOT NULL`): A unique identifier for each professor.
    - **`prof_last_name`** (`varchar(255)`): The last name of the professor. This field should ideally be `NOT NULL` for data integrity, as a last name is typically required.
    - **`prof_first_name`** (`varchar(255)`): The first name of the professor. This field can be `NULL`.
    - **`prof_middle_name`** (`varchar(255)`): The middle name of the professor. This field can be `NULL`.

---

#### `courses`

- **Description**: Stores information about course offerings. Each row represents a specific instance of a course offered in a particular year and quarter by a specific instructor.
- **Fields**:
    - **`course_offering_id`** (`varchar(255)`, **Primary Key**, `NOT NULL`): A unique identifier for each specific course offering. This is the primary key and links to other tables.
    - **`department`** (`varchar(255)`): The department offering the course (e.g., "Computer Science"). This field can be `NULL` but should ideally be `NOT NULL` for better data integrity.
    - **`course_id`** (`varchar(255)`): The course code or number (e.g., "CS101"). This field can be `NULL` but should ideally be `NOT NULL`.
    - **`year`** (`bigint`): The academic year in which the course is offered. This field can be `NULL` but should ideally be `NOT NULL`.
    - **`quarter`** (`varchar(255)`): The academic quarter in which the course is offered (e.g., "Fall", "Winter", "Spring", "Summer"). This field can be `NULL` but should ideally be `NOT NULL`.
    - **`instructor`** (`varchar(255)`): The name of the instructor for this specific course offering. While `prof_id` in `courses_professors` links to the `professors` table, this field serves as a direct reference to the instructor's name, especially important for unique identification of a course offering (e.g., "CS101 offered by Professor Smith in Fall 2024"). This field can be `NULL`.
    - **`total`** (`bigint`): The total capacity or number of seats available for this course offering. This field can be `NULL`.

---

#### `courses_professors`

- **Description**: A junction table that links specific course offerings to the professors who teach them. This many-to-many relationship allows a course offering to have multiple professors and a professor to teach multiple course offerings.
- **Fields**:
    - **`id`** (`char(64)`, **Primary Key**, `NOT NULL`): A unique identifier for each entry in this linking table, generated by hashing `course_offering_id` and `prof_id` (e.g., `SHA2(course_offering_id + prof_id)`).
    - **`prof_id`** (`varchar(255)`, **Foreign Key**, `MUL`, `NULLABLE`): References the `prof_id` in the `professors` table. This field is currently `NULLABLE` but should ideally be `NOT NULL` to ensure a valid link to a professor.
    - **`course_offering_id`** (`varchar(255)`, **Foreign Key**, `MUL`, `NULLABLE`): References the `course_offering_id` in the `courses` table. This field is currently `NULLABLE` but should ideally be `NOT NULL` to ensure a valid link to a course offering.

---

#### `enrollment_snapshots`

- **Description**: Stores historical enrollment data for specific course offerings at different points in time.
- **Fields**:
    - **`course_offering_id`** (`varchar(255)`, **Primary Key**, `NOT NULL`): References the `course_offering_id` in the `courses` table. Part of a composite primary key with `date`.
    - **`date`** (`varchar(255)`, **Primary Key**, `NOT NULL`): The date when the enrollment snapshot was taken. Part of a composite primary key with `course_offering_id`. Consider changing the `Type` to `DATE` or `DATETIME` for better date-time handling and indexing.
    - **`enrolled_ct`** (`bigint`): The number of students currently enrolled in the course offering at the time of the snapshot. This field can be `NULL`.
    - **`waitlist`** (`bigint`): The number of students on the waitlist for the course offering at the time of the snapshot. This field can be `NULL`.

---

#### `passtimes`

- **Description**: Stores registration pass times for different academic years and quarters.
- **Fields**:
    - **`id`** (`varchar(255)`, **Primary Key**, `NOT NULL`): A unique identifier for each passtime entry.
    - **`year`** (`bigint`): The academic year for which the passtime applies. This field can be `NULL` but should ideally be `NOT NULL`.
    - **`quarter`** (`varchar(255)`): The academic quarter for which the passtime applies. This field can be `NULL` but should ideally be `NOT NULL`.
    - **`passtag`** (`varchar(255)`, `MUL`): The tag or type of registration pass (e.g., "Prior", "First Pass", "Second Pass"). This field can be `NULL`.
    - **`passtime`** (`varchar(255)`): The actual date or time of the registration pass, in `YYYY-MM-DD` format. Consider changing the `Type` to `DATE` or `DATETIME` for better date handling. This field can be `NULL`.

---

#### `course_professor_comments`

- **Description**: Stores comments related to specific course offerings and their associated professors.
- **This table does not managed by data cleaning process.**
- **Fields**:
    - **`comment_id`** (`bigint`, **Primary Key**, `NOT NULL`, `AUTO_INCREMENT`): A unique, auto-incrementing identifier for each comment.
    - **`course_offering_id`** (`varchar(255)`, **Foreign Key**, `MUL`, `NOT NULL`): References the `course_offering_id` in the `courses` table.
    - **`prof_id`** (`varchar(255)`, **Foreign Key**, `MUL`, `NOT NULL`): References the `prof_id` in the `professors` table.
    - **`comment`** (`text`): The actual content of the comment. This field can be `NULL`.
    - **`created_at`** (`timestamp`, `DEFAULT CURRENT_TIMESTAMP`): created time stamp.
    - **`updated_at`** auto update timestamp when comment update.
    - **`like_ct`** (`int`, `DEFAULT 0`): The number of "likes" for the comment. This field can be `NULL`.
    - **`dislike_ct`** (`int`, `DEFAULT 0`): The number of "dislikes" for the comment. This field can be `NULL`.

## Create Process

#### Required parameters and test S3 connection

In [0]:
import json
import os
import uuid
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum, when, lit
from pyspark.sql.functions import sum, avg, max, min, count, countDistinct, first, last, mean, stddev, collect_list, collect_set, approx_count_distinct, expr
from pyspark.sql.functions import col, from_unixtime, to_timestamp, date_format, row_number
from pyspark.sql.functions import split, locate, explode, trim, substring, size
from pyspark.sql.functions import sha2, concat_ws


# essential parameters to connect AWS S3
AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
BUCKET_NAME = ""
REGION = ""

In [0]:
spark.conf.set("fs.s3a.access.key", AWS_ACCESS_KEY)
spark.conf.set("fs.s3a.secret.key", AWS_SECRET_KEY)

In [0]:
# 关于S3的基本参数
base_path = f"s3a://{BUCKET_NAME}/ucsd"
path_final_data = f"{base_path}/final/final"
path_final_table = f"{base_path}/final_table"

try:
    df = spark.read.csv(f"{path_final_data}", header=True, inferSchema=True)
    display(df.show(3))
    df.printSchema()
except Exception as e:
    print(f"Table read failed: {e}")

+--------------------+----------+-----+--------+-----------+----+-------+----------+---------+
|                prof|      date|total|waitlist|enrolled_ct|year|quarter|department|course_id|
+--------------------+----------+-----+--------+-----------+----+-------+----------+---------+
|Solomon; Amanda L...|2025-01-03|   -1|       0|          5|2025| Winter|      AAPI|      198|
|Solomon; Amanda L...|2025-01-02|   -1|       0|          5|2025| Winter|      AAPI|      198|
|Solomon; Amanda L...|2025-01-18|   -1|       0|          7|2025| Winter|      AAPI|      198|
+--------------------+----------+-----+--------+-----------+----+-------+----------+---------+
only showing top 3 rows

root
 |-- prof: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total: integer (nullable = true)
 |-- waitlist: integer (nullable = true)
 |-- enrolled_ct: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: string (nullable = true)
 |-- department: string (nullable = 

### Tool functions

In [0]:
# uuid generation function
# Deprecated
# Reason: uuid() runs once on each call, returning an independent value.
# However, due to Spark's lazy evaluation + immutability, each time we call withColumn(...) or other transformation operations,
# the original DataFrame is not modified; instead, a new DataFrame is returned.
# This new DataFrame will re-run the uuid generation function.
# Ultimately, this causes UUIDs to differ between the two DataFrames.
uuid_udf = F.udf(lambda: str(uuid.uuid4()), StringType())

# key generation function
def generate_id(df, columns, key_col_name):
    df = df.withColumn(key_col_name, sha2(concat_ws("||", *columns), 256))
    return df

# Splits professor names into first_name, last_name, middle_name
def split_prof_name(df):

    # Splits courses taught by multiple professors
    # Some courses are taught by multiple professors, in which case their names are joined by '&'
    # For example: Bafna; Vineet & Zhong; Sheng
    # This splits them into multiple rows.
    df = df.withColumn("prof", explode(split(col("prof"), "& "))) \
           .withColumn("prof", trim(col("prof")))

    # For professor names, the format is last_name; first_name middle_name (middle_name can be null)
    # This splits them into three columns.
    # If there's no middle name, the prof_middle_name column will be null.
    # If the professor is 'Staff', both prof_first_name and prof_last_name will be 'Staff'.

    df = df.withColumn("isStaff", col("prof") == lit("Staff"))

    df = df.withColumn(
        "prof_last_name",
        when(col("isStaff"), "Staff")
        .otherwise(trim(split(col("prof"), "; ").getItem(0)))
    ).withColumn(
        "first_middle_name",
        when(col("isStaff"), "Staff")
        .otherwise(trim(split(col("prof"), "; ").getItem(1)))
    )

    df = df.withColumn(
        "prof_first_name", 
        when(col("isStaff"), "Staff")
        .otherwise(split(col("first_middle_name"), " ", 2).getItem(0))
    ).withColumn(
        "prof_middle_name",
        when(col("isStaff"), lit(None))
        .otherwise(
            # Checks for middle name
            when(size(split(col("first_middle_name"), " ")) > 1, split(col("first_middle_name"), " ", 2).getItem(1))
            .otherwise(lit(None))
        )
    )
    
    df = df.drop("isStaff", "first_middle_name")

    return df

# Prepares data for the professors table
def create_prof_table_data(df_original):
    df = df_original
    df = df.select("prof").distinct()
    
    # Splits professor names
    df = split_prof_name(df)
    df = df.drop("prof").distinct()

    # Generates primary key column
    df = generate_id(df, ["prof_first_name", "prof_last_name", "prof_middle_name"], "prof_id")

    return df

# Prepares data for the Courses table
def create_courses_table_data(df):
    df = df.select("department", "course_id", "year", "quarter", "total", "prof").distinct()

    # Generates primary key
    df = generate_id(df, ["department", "course_id", "year", "quarter", "total", "prof"], "course_offering_id")
    return df

# Creates the Courses-Professors linking table
# The temporary JOIN column between the two tables is 'prof'
def create_courses_professors_table_data(courses, professors, registrations_original):

    # Gets courses_offering_id
    df = registrations_original.join(courses, on=["year", "quarter", "department", "course_id", "total", "prof"], how="inner")

    # Now, prof_first_name, prof_last_name, prof_middle_name columns are added
    df = split_prof_name(df)

    # Gets prof_id
    # Directly using on=["prof_first_name", "prof_last_name", "prof_middle_name"] would cause significant issues.
    # In SQL and Spark, NULL = NULL evaluates to False, and prof_middle_name contains NULL values. A direct join would filter out all professors without a middle name.
    # Therefore, a Null-Safe Join is required. Spark's NULL <=> NULL returns True.
    df = df.join(professors, on=[
        df.prof_first_name == professors.prof_first_name, 
        df.prof_last_name == professors.prof_last_name, 
        df.prof_middle_name.eqNullSafe(professors.prof_middle_name)
        # Alternatively: df['prof_middle_name'] <=> professors['prof_middle_name']
        ], 
        how="inner"
    )

    # Selects only the required two columns
    df = df.select("prof_id", "course_offering_id").distinct()

    # Adds primary key
    df_with_id = df.withColumn(
        "id",
        F.sha2(F.concat(col("course_offering_id"), col("prof_id")), 256)
    )
    df_final = df_with_id.select("id", "prof_id", "course_offering_id")

    return df_final

def create_enrollment_snapshots_table_data(registrations_original, courses):
    df = registrations_original
    
    # Gets courses_id
    df = df.join(courses, on=["year", "quarter", "department", "course_id", "prof", "total"], how="inner")

    # Selects only the required columns
    df = df.select("date", "waitlist", "enrolled_ct", "course_offering_id").distinct()

    return df

def clean_tables(professors, courses, courses_professors, enrollment_snapshots):

    # professors = professors.selectExpr("")
    
    # Converts 'prof' column to 'instructor' column
    # Format: ["prof_first_name prof_last_name", "prof_first_name prof_last_name"]
    courses = courses.withColumn(
        "instructor",
        F.transform(
            # First, split the professor string into an array by "&"
            # E.g., "Bafna; Vineet & Zhong; Sheng" -> ["Bafna; Vineet", "Zhong; Sheng"]
            F.split(F.col("prof"), "& "),

            # For each element in the array, i.e., the professor's name, transform it
            lambda p: when(
                # If it's 'Staff', follow the 'Professors' naming convention where both names are 'Staff'
                trim(p) == "Staff",
                lit("Staff Staff")
            ).otherwise(
                # Otherwise, process "last_name; firstname" to "first_name last_name"
                F.concat_ws(" ", 
                    # First, split by semicolon, then split by space and take the first part, which is the first_name
                    F.split(F.trim(F.split(p, "; ").getItem(1)), " ").getItem(0),
                    trim(F.split(p, ";").getItem(0))
                )
            )
        )
    ).drop("prof")

    # transform the instructor from array<string> to string
    courses = courses.withColumn("instructor", concat_ws(",", col("instructor")))

    return professors, courses, courses_professors, enrollment_snapshots

### Transfer Each Table

In [0]:
data_final = spark.read.csv(f"{path_final_data}", header=True, inferSchema=True)

In [0]:
df_professors = create_prof_table_data(data_final)
df_professors.show(3)

+--------------+---------------+----------------+--------------------+
|prof_last_name|prof_first_name|prof_middle_name|             prof_id|
+--------------+---------------+----------------+--------------------+
|           Som|        Brandon|               D|29d8e421bec3ce8b9...|
|  Sanchez Cruz|          Jorge|            null|42c8dd041bb069f27...|
|    Schurmeier|       Kimberly|            null|d3dc139bd6cc4c799...|
+--------------+---------------+----------------+--------------------+
only showing top 3 rows



In [0]:
df_courses = create_courses_table_data(data_final)
df_courses.show(3)

+----------+---------+----+-------+-----+--------------------+--------------------+
|department|course_id|year|quarter|total|                prof|  course_offering_id|
+----------+---------+----+-------+-----+--------------------+--------------------+
|       AAS|       11|2025| Winter|   68|Butler; Elizabeth...|f18fb01db40425f1d...|
|      AESE|      241|2025| Winter|   35|Erat; Sanjiv & Wa...|8ef5ad892185ad15a...|
|       AAS|       10|2025| Winter|   68|Butler; Elizabeth...|c85a26fe19ff326df...|
+----------+---------+----+-------+-----+--------------------+--------------------+
only showing top 3 rows



In [0]:
df_courses_professors = create_courses_professors_table_data(df_courses, df_professors, data_final)
df_courses_professors.show(3, truncate=False)

+--------------------+--------------------+--------------------+
|                  id|             prof_id|  course_offering_id|
+--------------------+--------------------+--------------------+
|ecc931acd26217641...|d3c5ad366acec8d0c...|4704e7abe797a4a6a...|
|1078ca23e131f332d...|eae519d5991bc343a...|6d1f02cafc5ab40a2...|
|c8bb5c574dc321094...|642c39d80494ad810...|94c23ae8deac19617...|
+--------------------+--------------------+--------------------+
only showing top 3 rows



In [0]:
df_enrollment_snapshots = create_enrollment_snapshots_table_data(data_final, df_courses)
df_enrollment_snapshots.show(3)

+----------+--------+-----------+--------------------+
|      date|waitlist|enrolled_ct|  course_offering_id|
+----------+--------+-----------+--------------------+
|2024-11-13|       0|          2|6d1f02cafc5ab40a2...|
|2024-11-15|       0|          2|6d1f02cafc5ab40a2...|
|2024-11-20|       0|          2|6d1f02cafc5ab40a2...|
+----------+--------+-----------+--------------------+
only showing top 3 rows



In [0]:
# final clean
df_professors, df_courses, df_courses_professors, df_enrollment_snapshots = clean_tables(df_professors, df_courses, df_courses_professors, df_enrollment_snapshots)

## Store the table data

In [0]:
path_table_professors = f"{path_final_table}/professors"
path_table_courses = f"{path_final_table}/courses"
path_table_courses_professors = f"{path_final_table}/courses_professors"
path_table_enrollment_snapshots = f"{path_final_table}/enrollment_snapshots"

# write to s3
df_professors.coalesce(1).write.csv(path_table_professors, mode="overwrite", header=True)
df_courses.coalesce(1).write.csv(path_table_courses, mode="overwrite", header=True)
df_courses_professors.coalesce(1).write.csv(path_table_courses_professors, mode="overwrite", header=True)
df_enrollment_snapshots.coalesce(1).write.csv(path_table_enrollment_snapshots, mode="overwrite", header=True)

## Conclusion
We now have four datasets, each stored in a separate folder under {base_path}/final_table/.